# 03 — Stage-2 Unified Multi-Modality Multi-Task Training (`Pan-Organ Net V2`)

Initializes from Stage-1 SSL weights (`stage1_ssl_ma_ugm/best.pt`), registers frozen BiomedCLIP ontology prompt embeddings, and jointly optimizes multi-organ segmentation (`Dice + Focal + Boundary`), multi-pathology classification (`Asymmetric Focal` + `Focal CE`), `SACR`, and `MT-MoE`.


In [ ]:
import os, sys, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive/ResearchPilot_PanOrgan/V2')
CODE_DIR = DRIVE_ROOT / 'code'
SHARDS_DIR = DRIVE_ROOT / 'preprocessed_shards'
CKPT_DIR = DRIVE_ROOT / 'checkpoints'
EXP_DIR = DRIVE_ROOT / 'experiments'
for d in [SHARDS_DIR, CKPT_DIR, EXP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Unpack code package onto local fast SSD if uploaded as tarball, or install directly
if (CODE_DIR / 'V2_code_package.tar.gz').exists():
    Path('/content/panorgan_v2').mkdir(parents=True, exist_ok=True)
    !tar -xzf "{CODE_DIR}/V2_code_package.tar.gz" -C /content/panorgan_v2

PKG_ROOT = Path('/content/panorgan_v2') if Path('/content/panorgan_v2/pyproject.toml').exists() else CODE_DIR
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))
!pip install -q --no-deps -e "{PKG_ROOT}"
!pip install -q timm nibabel pydicom pyarrow scikit-learn scipy PyYAML tqdm open-clip-torch kaggle

import importlib
importlib.invalidate_caches()
import torch
from panorgan.config import load_config, detect_runtime_profile, seed_everything

cfg = load_config(PKG_ROOT / 'configs' / 'v2_panorgan.yaml')
seed_everything(cfg['project']['seed'], cfg['project']['deterministic'])
runtime = detect_runtime_profile(cfg)
print(f'Active Runtime -> GPU: {runtime.gpu_name} | Tier: {runtime.tier} | Batch: {runtime.batch_size}x{runtime.grad_accum} | AMP: {runtime.amp_dtype}')


In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
from panorgan.data import BalancedModalityBatchSampler, PanOrganShardDataset, ShardCacheManager
from panorgan.models import PanOrganNetV2
from panorgan.engine import ColabResilientTrainer

manifest_df = pd.read_parquet(SHARDS_DIR / 'master_manifest.parquet')
train_df = manifest_df[manifest_df['split'] == 'train'].reset_index(drop=True)
val_df = manifest_df[manifest_df['split'] == 'val'].reset_index(drop=True)

mod_map = {k: int(v['id']) for k, v in cfg['modalities'].items()}
cache_mgr = ShardCacheManager(
    drive_shards_root=SHARDS_DIR,
    local_cache_dir=Path(cfg['storage']['local_cache_dir']),
    max_cache_gb=float(cfg['storage']['max_local_cache_gb']),
)
cache_mgr.warmup_all_shards()

train_ds = PanOrganShardDataset(train_df, cache_mgr, mod_map, target_hw=(256, 256), augment=True, mnar_truncation=0.20)
val_ds = PanOrganShardDataset(val_df, cache_mgr, mod_map, target_hw=(256, 256), augment=False, mnar_truncation=0.0)

sampler = BalancedModalityBatchSampler(
    train_df['dataset'].tolist(),
    batch_size=runtime.batch_size,
    temperature_alpha=cfg['training']['stage2_multitask']['sampling_temperature'],
)
train_loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=cfg['storage']['num_workers'], pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=runtime.batch_size, shuffle=False, num_workers=cfg['storage']['num_workers'], pin_memory=True)

model = PanOrganNetV2.from_config(cfg)

# Load Stage-1 SSL pretrained encoder weights if present
ssl_ckpt = CKPT_DIR / 'stage1_ssl_ma_ugm' / 'best.pt'
if ssl_ckpt.exists():
    state = torch.load(ssl_ckpt, map_location='cpu', weights_only=False)
    missing, unexpected = model.load_state_dict(state['model_state_dict'], strict=False)
    print(f'Loaded Stage-1 SSL checkpoint. Missing: {len(missing)}, Unexpected: {len(unexpected)}')

# Optional: precompute frozen BiomedCLIP text embeddings once for all unique prompts
try:
    import open_clip
    clip_model, _ = open_clip.create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
    tokenizer = open_clip.get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
    clip_model.eval()
    unique_prompts = sorted(manifest_df['prompt'].dropna().unique().tolist())
    with torch.no_grad():
        tokens = tokenizer(unique_prompts)
        embs = clip_model.encode_text(tokens)
        embs = embs / embs.norm(dim=-1, keepdim=True)
    model.prompt_encoder.register_precomputed({p: e for p, e in zip(unique_prompts, embs)})
    del clip_model
    torch.cuda.empty_cache()
    print(f'Cached frozen BiomedCLIP embeddings for {len(unique_prompts)} clinical prompts.')
except Exception as exc:
    print(f'[INFO] Using deterministic prompt encoder fallback ({exc}).')

s2 = cfg['training']['stage2_multitask']
trainer = ColabResilientTrainer(
    model=model,
    runtime=runtime,
    checkpoint_dir=CKPT_DIR / 'stage2_panorgan_v2_full',
    lr=float(s2['lr']),
    backbone_lr_mult=float(s2['backbone_lr_mult']),
    weight_decay=float(s2['weight_decay']),
    max_grad_norm=float(s2['max_grad_norm']),
    seg_loss_weight=float(s2['seg_loss_weight']),
    cls_loss_weight=float(s2['cls_loss_weight']),
    sacr_loss_weight=float(s2['sacr_loss_weight']),
    checkpoint_every_steps=int(s2['checkpoint_every_steps']),
    patience=int(s2['patience']),
)
history = trainer.fit(train_loader, val_loader, num_epochs=int(s2['epochs']), mode='multitask', resume=True)
print('Stage-2 Multi-Task Training Complete! Best Val Loss:', trainer.best_val_loss)
